In [1]:
from pathlib import Path
from itertools import combinations
from math import comb
import pandas as pd
import unicodedata
import re
import math


In [2]:
DATA_PATH = Path("C:/Users/chuni/PycharmProjects/vacancies_hh/it_vacancies_2024.csv")
MIN_SALARIES = 30
UNKNOWN_GROSS_AS_NET = True
COLUMNS = [
    "source", "vacancy_id", "snapshot_date", "name", "role", "area",
    "experience", "salary_from", "salary_to", "currency", "gross", "key_skills",
]


#### Читаем файл


In [3]:
df = pd.read_csv(DATA_PATH, usecols=COLUMNS, sep=",", encoding="utf-8-sig", dtype="string")
print(f"CSV: {DATA_PATH.stat().st_size / 2**20:.1f} MiB")
print(f"Строк: {len(df):,}; столбцов: {len(df.columns)}")
print(f"DataFrame: {df.memory_usage(deep=True).sum() / 2**20:.1f} MiB")


CSV: 808.4 MiB
Строк: 229,388; столбцов: 12
DataFrame: 213.6 MiB


#### Фильтр своего варианта


In [4]:
PATTERN_1C = r"(?<!\w)1\s*[сc](?![a-zа-яё])"
PATTERN_DEV = r"программист|разработчик|developer|programmer"

def select_variant(frame):
    mask = (
        frame["role"].str.strip().eq("Программист, разработчик")
        & frame["name"].str.contains(PATTERN_1C, case=False, na=False)
        & frame["name"].str.contains(PATTERN_DEV, case=False, na=False)
    )
    return frame.loc[mask.fillna(False)].copy()


#### Применяем фильтр и проверяем ID

In [5]:
df_1c = select_variant(df)
duplicate_count = df_1c.duplicated(["source", "vacancy_id"]).sum()
missing_id_count = df_1c[["source", "vacancy_id"]].isna().any(axis=1).sum()
print("Вакансий варианта:", len(df_1c))
print("Повторов ID:", duplicate_count, "Пропущенных ID:", missing_id_count)
assert duplicate_count == 0 and missing_id_count == 0, "Сначала разберите повторы или пропуски ID."
print("С Битрикс в названии (оставлены):",
      df_1c["name"].str.contains(r"битрикс|bitrix", case=False, na=False).sum())


Вакансий варианта: 18943
Повторов ID: 0 Пропущенных ID: 0
С Битрикс в названии (оставлены): 552


#### Нормализация навыков


In [6]:
INVISIBLE = r"[\u200b\u200c\u200d\ufeff\u2060]"
ALIASES = {
    "1с erp": "1с:erp",
    "1с зуп": "1с:зарплата и управление персоналом",
    "1с:зуп": "1с:зарплата и управление персоналом",
    "1с упп": "1с:управление производственным предприятием",
    "1с:упп": "1с:управление производственным предприятием",
    "1с ух": "1с:управление холдингом",
    "1с:ух": "1с:управление холдингом",
    "ms sql": "microsoft sql server",
    "ms sql server": "microsoft sql server",
    "с#": "c#",
}


#### Функции очистки


In [7]:
def normalize_skill(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(INVISIBLE, "", text)
    text = re.sub(r"\s+", " ", text).strip().casefold()
    text = re.sub(r"^(?:[-–—•·*]|\.)\s+", "", text)
    text = text.rstrip(" ,;.")
    text = re.sub(PATTERN_1C, "1с", text)
    text = re.sub(r"\s*:\s*", ":", text)
    return ALIASES.get(text, text)

def normalize_skills(value):
    if pd.isna(value):
        return []
    return sorted({skill for item in value.split(";") if (skill := normalize_skill(item))})


#### Проверяем на примерах


In [8]:
for example in ["Docker;", "  docker", "Git\u200b", ". 1C программирование", "С#", "C++", ".NET"]:
    print(f"{example!r:30} -> {normalize_skill(example)!r}")


'Docker;'                      -> 'docker'
'  docker'                     -> 'docker'
'Git\u200b'                    -> 'git'
'. 1C программирование'        -> '1с программирование'
'С#'                           -> 'c#'
'C++'                          -> 'c++'
'.NET'                         -> '.net'


#### Зарплата по правилу c практики


In [9]:
def prepare_salary(frame):
    result = frame.copy()
    raw = result[["salary_from", "salary_to"]].replace(r"^\s*$", pd.NA, regex=True)
    bounds = pd.DataFrame({
        col: pd.to_numeric(raw[col], errors="coerce").astype("Float64")
        for col in raw.columns
    })
    lower, upper = bounds["salary_from"], bounds["salary_to"]
    result["salary_invalid"] = (
        (raw.notna() & bounds.isna()).any(axis=1)
        | bounds.le(0).fillna(False).any(axis=1)
        | bounds.isin([float("inf"), -float("inf")]).any(axis=1)
        | lower.gt(upper).fillna(False)
    )
    result["salary_kind"] = "не указана"
    result.loc[lower.notna(), "salary_kind"] = "только от"
    result.loc[upper.notna(), "salary_kind"] = "только до"
    result.loc[lower.notna() & upper.notna(), "salary_kind"] = "диапазон"
    result.loc[result["salary_invalid"], "salary_kind"] = "некорректная"
    result["gross_normalized"] = result["gross"].astype("string").str.strip().str.lower().replace("", pd.NA)
    result["currency_normalized"] = result["currency"].str.strip().str.upper()
    gross = result["gross_normalized"]
    known = gross.isin(["true", "false"])
    accepted = known | (gross.isna() & UNKNOWN_GROSS_AS_NET)
    eligible = (
        result["currency_normalized"].eq("RUR") & lower.notna()
        & ~result["salary_invalid"] & accepted
    ).fillna(False)
    net = lower.where(eligible)
    before_tax = gross.eq("true").fillna(False)
    result["net"] = net.where(~before_tax, net * 87 // 100)
    return result


#### Общая подготовка одной таблицы или чанка


In [10]:
def prepare_vacancies(frame):
    result = select_variant(frame)
    result["skills_normalized"] = result["key_skills"].apply(normalize_skills)
    result["n_skills"] = result["skills_normalized"].str.len()
    return prepare_salary(result)


#### Применяем подготовку


In [11]:
df_1c = prepare_vacancies(df)
df_salary = df_1c.loc[df_1c["net"].notna()].copy()


#### Кто указывает зарплату и какие суммы получились


In [12]:
kind_counts = df_1c["salary_kind"].value_counts()
print(pd.DataFrame({"Вакансий": kind_counts, "Доля, %": (kind_counts / len(df_1c) * 100).round(1)}).to_string())


             Вакансий  Доля, %
salary_kind                   
не указана       7410     39.1
только от        5833     30.8
диапазон         4483     23.7
только до        1217      6.4


#### Валюта и признак gross


In [13]:
paid = df_1c.loc[df_1c["salary_kind"].ne("не указана")]
print("\nВалюты среди указавших зарплату:\n", paid["currency_normalized"].value_counts(dropna=False).to_string())
print("\nGross среди указавших зарплату:\n", paid["gross_normalized"].value_counts(dropna=False).to_string())



Валюты среди указавших зарплату:
 currency_normalized
RUR    11512
BYR        8
USD        7
KZT        6

Gross среди указавших зарплату:
 gross_normalized
false    8042
true     3488
<NA>        3


#### Зарплата на руки


In [14]:
unknown = df_salary["gross_normalized"].isna()
print("\nВакансий с net:", len(df_salary), "Из них с неизвестным gross:", unknown.sum())
print("Медиана net:", df_salary["net"].median())
print("Медиана без неизвестного gross:", df_salary.loc[~unknown, "net"].median())
print("С навыками и net:", df_salary["n_skills"].gt(0).sum())
print("\nРаспределение net:\n", df_salary["net"].describe(percentiles=[.01, .5, .99]).round(0).to_string())



Вакансий с net: 10301 Из них с неизвестным gross: 0
Медиана net: 130500.0
Медиана без неизвестного gross: 130500.0
С навыками и net: 7031

Распределение net:
 count     10301.0
mean     134894.0
std       66175.0
min          39.0
1%        20000.0
50%      130500.0
99%      300000.0
max      460000.0


#### Проверяем крайние суммы


In [15]:
cols = ["vacancy_id", "name", "salary_from", "salary_to", "gross_normalized", "net"]
print("\nПять минимальных сумм (для проверки):\n", df_salary.nsmallest(5, "net")[cols].to_string(index=False))
print("\nПять максимальных сумм:\n", df_salary.nlargest(5, "net")[cols].to_string(index=False))



Пять минимальных сумм (для проверки):
 vacancy_id                                                     name salary_from salary_to gross_normalized   net
  91846309                                           Программист 1С          45        50             true  39.0
 104181472 Web-разработчик 1c Битрикс программист PHP Middle/Senior          40    120000            false  40.0
  89863577                                           Программист 1С          70      <NA>             true  60.0
 101056751                                           Программист 1С          80    100000            false  80.0
 104118435                                           Программист 1С          80    100000            false  80.0

Пять максимальных сумм:
 vacancy_id                                     name salary_from salary_to gross_normalized       net
  94212812          Ведущий программист 1С (Москва)      460000    517000            false  460000.0
  95395666 Ведущий программист 1С, г. Москва (офис)   

#### Навыки и рост числа сочетаний


In [16]:
raw_skills = df_1c["key_skills"].dropna().str.split(";").explode().str.strip()
raw_skills = raw_skills[raw_skills.ne("")]
skill_counts = df_1c["skills_normalized"].explode().dropna().value_counts()
print("Написаний до:", raw_skills.nunique(), "Навыков после:", len(skill_counts))
print("С невидимыми символами:", raw_skills.str.contains(INVISIBLE, regex=True).sum())
print("Вакансий с навыками:", df_1c["n_skills"].gt(0).sum())
print("Максимум навыков:", df_1c["n_skills"].max())


Написаний до: 2486 Навыков после: 2181
С невидимыми символами: 0
Вакансий с навыками: 12409
Максимум навыков: 30


#### Сколько получится пар и троек


In [17]:
growth = pd.DataFrame([
    {"k": k, "Вакансий с k навыками или больше": int(df_1c["n_skills"].ge(k).sum()),
     "Промежуточных сочетаний": sum(comb(int(n), k) for n in df_1c["n_skills"] if n >= k)}
    for k in (1, 2, 3)
])
print("\n", growth.to_string(index=False))
print("\nТоп-15 навыков:\n", skill_counts.head(15).to_string())



  k  Вакансий с k навыками или больше  Промежуточных сочетаний
 1                             12409                    68363
 2                             11375                   231925
 3                             10228                   694183

Топ-15 навыков:
 skills_normalized
1с программирование                            9412
1с:предприятие 8                               4837
1с:бухгалтерия                                 3586
1с:управление торговлей                        2856
обновление конфигурации 1с                     2486
erp-системы на базе 1с                         2302
1с:зарплата и управление персоналом            2292
1с:документооборот                             2083
1с:предприятие                                 1627
создание конфигурации 1с                       1576
1с:управление производственным предприятием    1370
1с:комплексная автоматизация                   1098
1с:зарплата и кадры                            1065
1с:erp                                

#### Контрольная статистика Pandas: навыки, пары и тройки


In [18]:
def to_long(prepared):
    base = prepared.reset_index(drop=True)
    skills = base["skills_normalized"].explode().dropna()
    return pd.DataFrame({
        "row": skills.index,
        "skill": skills.to_numpy(),
        "net": base["net"].reindex(skills.index).to_numpy(),
    }).drop_duplicates(["row", "skill"])


#### Формируем сочетания внутри вакансии


In [19]:
def combos(long, k):
    if k not in (1, 2, 3):
        raise ValueError("k должен быть 1, 2 или 3")
    # row - одна вакансия, ни одна комбинация не соединяет разные вакансии.
    grouped = long.groupby("row", sort=False).agg(
        skills=("skill", list), net=("net", "first")
    )
    records = (
        (*stack, row.net)
        for row in grouped.itertuples(index=False)
        for stack in combinations(sorted(set(row.skills)), k)
    )
    table = pd.DataFrame.from_records(records, columns=[f"s{i}" for i in range(1, k + 1)] + ["net"])
    table["net"] = pd.to_numeric(table["net"], errors="coerce")
    return table


#### Считаем частоты и медианы


In [20]:
def stats(table, k):
    keys = [f"s{i}" for i in range(1, k + 1)]
    result = table.groupby(keys, as_index=False).agg(
        vacancies=("net", "size"), salaries=("net", "count"), median_net=("net", "median")
    )
    result.loc[result["salaries"] < MIN_SALARIES, "median_net"] = float("nan")
    return result.sort_values(["vacancies"] + keys, ascending=[False] + [True] * k).reset_index(drop=True)


#### Запускаем расчёт для k = 1, 2, 3


In [21]:
long = to_long(df_1c)
results_pandas = {}
for k in (1, 2, 3):
    expanded = combos(long, k)
    expected = int(growth.loc[growth["k"].eq(k), "Промежуточных сочетаний"].iloc[0])
    assert len(expanded) == expected
    results_pandas[k] = stats(expanded, k)
    assert int(results_pandas[k]["vacancies"].sum()) == expected
    print(f"\nk={k}: разных сочетаний — {len(results_pandas[k]):,}")
    print(results_pandas[k].head(5).to_string(index=False))
    del expanded



k=1: разных сочетаний — 2,181
                        s1  vacancies  salaries  median_net
       1с программирование       9412      5405    130500.0
          1с:предприятие 8       4837      3025    130500.0
            1с:бухгалтерия       3586      2142    130000.0
   1с:управление торговлей       2856      1850    130500.0
обновление конфигурации 1с       2486      1501    130500.0

k=2: разных сочетаний — 30,224
                 s1                         s2  vacancies  salaries  median_net
1с программирование           1с:предприятие 8       4084      2574    130500.0
1с программирование             1с:бухгалтерия       2884      1757    130000.0
1с программирование    1с:управление торговлей       2279      1537    130500.0
1с программирование обновление конфигурации 1с       2209      1340    130500.0
     1с:бухгалтерия           1с:предприятие 8       1931      1274    130500.0

k=3: разных сочетаний — 195,046
                 s1               s2                            

#### Территории и опыт


In [22]:
def group_summary(prepared, column):
    data = prepared.assign(has_skills=prepared["n_skills"].gt(0))
    result = data.groupby(column, dropna=False).agg(
        vacancies=("vacancy_id", "size"), with_skills=("has_skills", "sum"),
        salaries=("net", "count"), median_net=("net", "median")
    )
    result["salary_share_pct"] = (result["salaries"] / result["vacancies"] * 100).round(1)
    result.loc[result["salaries"] < MIN_SALARIES, "median_net"] = float("nan")
    return result.sort_values("vacancies", ascending=False)


#### Сравниваем территории и опыт


In [23]:
regions = group_summary(df_1c, "area")
print("Территории:\n", regions.head(10).to_string())
print("\nОпыт:\n", group_summary(df_1c, "experience").to_string())


Территории:
                  vacancies  with_skills  salaries  median_net  salary_share_pct
area                                                                           
Москва                5248         3456      2284    180000.0              43.5
Санкт-Петербург       1703         1159       896    150000.0              52.6
Екатеринбург           884          595       546    130500.0              61.8
Краснодар              551          377       338    140000.0              61.3
Новосибирск            543          359       294    130500.0              54.1
Нижний Новгород        499          326       267    150000.0              53.5
Казань                 450          321       248    134850.0              55.1
Красноярск             441          303       333     80000.0              75.5
Ростов-на-Дону         387          245       225    120000.0              58.1
Челябинск              382          246       222    125500.0              58.1

Опыт:
                    

#### Свой MapReduce: mapper


In [24]:
JOBS_DIR = DATA_PATH.parent / "jobs"
JOBS_DIR.mkdir(exist_ok=True)
MAPPER_PATH = JOBS_DIR / "mapper.py"


#### Подключаем mapper


In [25]:
import csv
import importlib.util

spec = importlib.util.spec_from_file_location("vacancies_mapper", MAPPER_PATH)
mapper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mapper)


#### Проверяем одну вакансию


In [26]:
example = {
    "source": "hh", "vacancy_id": "example", "role": "Программист, разработчик",
    "name": "Разработчик 1С", "key_skills": "1C ERP; SQL; sql; Git\u200b",
    "salary_from": "100000", "salary_to": "150000", "currency": "RUR", "gross": "true",
}
mapper.prepare_row(example, UNKNOWN_GROSS_AS_NET)


{'source': 'hh',
 'vacancy_id': 'example',
 'skills': ['1с:erp', 'git', 'sql'],
 'net': 87000.0}

#### Читаем CSV последовательно и показываем длинную таблицу


In [27]:
preview = []
vacancies_shown = 0
with DATA_PATH.open(encoding="utf-8-sig", newline="") as stream:
    for row_number, row in enumerate(csv.DictReader(stream)):
        prepared = mapper.prepare_row(row, UNKNOWN_GROSS_AS_NET)
        if prepared is None or not prepared["skills"]:
            continue
        for skill in prepared["skills"]:
            preview.append({"row": row_number, "vacancy_id": prepared["vacancy_id"],
                            "skill": skill, "net": prepared["net"]})
        vacancies_shown += 1
        if vacancies_shown == 5:
            break
display(pd.DataFrame(preview))


,row,vacancy_id,skill,net
0,31,106356656,1с программирование,NaN
1,31,106356656,1с:предприятие 8,NaN
2,31,106356656,erp-системы на базе 1с,NaN
3,31,106356656,разработка технических заданий,NaN
4,31,106356656,создание конфигурации 1с,NaN
5,45,106357315,инициативность,139200.0
6,45,106357315,исполнительность,139200.0
7,45,106357315,ответственность,139200.0
8,45,106357315,развитые коммуникативные навыки,139200.0
9,50,106496717,1с программирование,200000.0


#### Reduce: частоты и медиана

In [28]:
REDUCER_PATH = JOBS_DIR / "reducer_stats.py"


#### Запуск Map -> Shuffle -> Reduce

In [29]:
RUNNER_PATH = DATA_PATH.parent / "step_mapreduce.py"


#### Подключение к RustFS: чтение из объектного хранилища


In [30]:
%pip install boto3


Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
from getpass import getpass

os.environ["S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
MR_INPUT = "s3://dataset/it_vacancies_2024.csv"

# Ключи вводятся скрыто, не сохраняются в тексте ноутбука
os.environ["AWS_ACCESS_KEY_ID"] = getpass("Access key RustFS: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("Secret key RustFS: ")


In [32]:
# Проверяем доступ и размер объекта, ничего не скачивая целиком
import importlib.util

spec = importlib.util.spec_from_file_location("mr_s3_check", RUNNER_PATH)
mr_s3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mr_s3)
s3_info = mr_s3.describe_source(MR_INPUT)
print("Источник:", s3_info["location"])
print("Размер:", s3_info["size"], "байт")
print("ETag:", s3_info["etag"])
print("Начало первого блока:")
print(mr_s3.get_range(s3_info, 0, min(300, s3_info["size"])).decode("utf-8-sig", errors="replace"))


Источник: s3://dataset/it_vacancies_2024.csv
Размер: 847619384 байт
ETag: "be9c1b4b3f57ff0e9771004c2a5fe78b-51"
Начало первого блока:
source,vacancy_id,status,snapshot_date,published_at,area,employer_id,employer_inn,employer_ogrn,employer_name,name,role,sphere,salary_from,salary_to,currency,gross,experience,schedule,employment,key_skills,description,url
hh,106654617,not_active,2024-09-03,2024-09-02,Краснодар,4105707,,1197


#### Навыки и пары: запуск MapReduce


In [33]:
import subprocess
import sys

command = [sys.executable, str(RUNNER_PATH), MR_INPUT, "1", "2",
           "--min-salaries", str(MIN_SALARIES),
           "--unknown-gross-as-net", str(int(UNKNOWN_GROSS_AS_NET))]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding="utf-8") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("MapReduce завершился с ошибкой — см. вывод выше")


Источник: s3://dataset/it_vacancies_2024.csv; размер 847,619,384 байт
CSV проверен: 229388 записей, переносов внутри записей нет
K=1: map-задач 13, вхождений 68,363, строк shuffle 68,363, shuffle 2.6 МиБ, редьюсеров 8
  1с программирование: вакансий 9412, зарплат 5405, медиана 130500.0
  1с:предприятие 8: вакансий 4837, зарплат 3025, медиана 130500.0
  1с:бухгалтерия: вакансий 3586, зарплат 2142, медиана 130000.0
  1с:управление торговлей: вакансий 2856, зарплат 1850, медиана 130500.0
  обновление конфигурации 1с: вакансий 2486, зарплат 1501, медиана 130500.0
K=2: map-задач 13, вхождений 231,925, строк shuffle 231,925, shuffle 15.9 МиБ, редьюсеров 8
  1с программирование|1с:предприятие 8: вакансий 4084, зарплат 2574, медиана 130500.0
  1с программирование|1с:бухгалтерия: вакансий 2884, зарплат 1757, медиана 130000.0
  1с программирование|1с:управление торговлей: вакансий 2279, зарплат 1537, медиана 130500.0
  1с программирование|обновление конфигурации 1с: вакансий 2209, зарплат 1340, 

#### Тройки


In [34]:
# Отдельный запуск только для троек
import subprocess
import sys

command = [sys.executable, str(RUNNER_PATH), MR_INPUT, "3",
           "--min-salaries", str(MIN_SALARIES),
           "--unknown-gross-as-net", str(int(UNKNOWN_GROSS_AS_NET))]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding="utf-8") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Расчёт троек завершился с ошибкой — см. вывод выше")


Источник: s3://dataset/it_vacancies_2024.csv; размер 847,619,384 байт
CSV проверен: 229388 записей, переносов внутри записей нет
K=3: map-задач 13, вхождений 694,183, строк shuffle 694,183, shuffle 68.1 МиБ, редьюсеров 8
  1с программирование|1с:бухгалтерия|1с:предприятие 8: вакансий 1611, зарплат 1070, медиана 130500.0
  1с программирование|1с:предприятие 8|1с:управление торговлей: вакансий 1308, зарплат 941, медиана 130500.0
  1с программирование|1с:предприятие 8|обновление конфигурации 1с: вакансий 1254, зарплат 836, медиана 130500.0
  1с программирование|1с:бухгалтерия|1с:управление торговлей: вакансий 1163, зарплат 812, медиана 130000.0
  1с программирование|1с:бухгалтерия|1с:зарплата и управление персоналом: вакансий 1053, зарплат 676, медиана 130500.0

Свой MapReduce: 31.74 с; пик памяти 0.67 ГиБ (private bytes, процесс и потомки)
K=3: стеков 195,046, вхождений вакансий 694,183, зарплат 444,603, сумма медиан 166,839,902.0
Результаты: C:\Users\chuni\PycharmProjects\vacancies_hh\m

#### Читаем результаты MapReduce


In [35]:
import json
from urllib.parse import unquote

results_mr = {}
for k in (1, 2, 3):
    info = json.loads((DATA_PATH.parent / "mr_work" / f"latest-k{k}.json").read_text(encoding="utf-8"))
    folder = Path(info["directory"])
    run_info = json.loads((folder.parent / "summary.json").read_text(encoding="utf-8"))
    assert run_info["input"] == MR_INPUT, "Результат от другого источника: сначала запусти расчёт из RustFS"
    assert (folder / "_SUCCESS").exists()
    records = []
    for part in sorted(folder.glob("part-[0-9][0-9][0-9][0-9][0-9]")):
        with part.open(encoding="utf-8") as stream:
            for line in stream:
                key, n, paid, median = line.rstrip("\n").split("\t")
                skills = [unquote(skill) for skill in key.split("|")]
                records.append(skills + [int(n), int(paid), float(median) if median else float("nan")])
    keys = [f"s{i}" for i in range(1, k+1)]
    results_mr[k] = pd.DataFrame(records, columns=keys + ["vacancies", "salaries", "median_net"])
    display(results_mr[k].sort_values("vacancies", ascending=False).head(10))


,s1,vacancies,salaries,median_net
1114,1с программирование,9412,5405,130500.0
1957,1с:предприятие 8,4837,3025,130500.0
22,1с:бухгалтерия,3586,2142,130000.0
34,1с:управление торговлей,2856,1850,130500.0
458,обновление конфигурации 1с,2486,1501,130500.0
1423,erp-системы на базе 1с,2302,1127,150000.0
848,1с:зарплата и управление персоналом,2292,1270,150000.0
1684,1с:документооборот,2083,1231,140000.0
1132,1с:предприятие,1627,1013,120000.0
257,создание конфигурации 1с,1576,1028,140000.0


,s1,s2,vacancies,salaries,median_net
11418,1с программирование,1с:предприятие 8,4084,2574,130500.0
11410,1с программирование,1с:бухгалтерия,2884,1757,130000.0
26446,1с программирование,1с:управление торговлей,2279,1537,130500.0
176,1с программирование,обновление конфигурации 1с,2209,1340,130500.0
26914,1с:бухгалтерия,1с:предприятие 8,1931,1274,130500.0
22680,1с программирование,erp-системы на базе 1с,1879,951,150000.0
22655,1с программирование,1с:документооборот,1705,1012,133425.0
7596,1с программирование,1с:зарплата и управление персоналом,1687,979,139200.0
27333,1с:предприятие 8,1с:управление торговлей,1503,1067,130500.0
218,1с программирование,создание конфигурации 1с,1447,944,140000.0


,s1,s2,s3,vacancies,salaries,median_net
171220,1с программирование,1с:бухгалтерия,1с:предприятие 8,1611,1070,130500.0
171540,1с программирование,1с:предприятие 8,1с:управление торговлей,1308,941,130500.0
123071,1с программирование,1с:предприятие 8,обновление конфигурации 1с,1254,836,130500.0
24822,1с программирование,1с:бухгалтерия,1с:управление торговлей,1163,812,130000.0
146892,1с программирование,1с:бухгалтерия,1с:зарплата и управление персоналом,1053,676,130500.0
24904,1с программирование,1с:документооборот,1с:предприятие 8,998,640,140000.0
24862,1с программирование,1с:бухгалтерия,обновление конфигурации 1с,974,624,147900.0
49386,1с программирование,1с:зарплата и управление персоналом,1с:предприятие 8,914,572,150000.0
147197,1с программирование,1с:предприятие 8,erp-системы на базе 1с,885,516,150000.0
5285,1с:бухгалтерия,1с:предприятие 8,1с:управление торговлей,879,626,130500.0


#### DuckDB: независимый SQL-расчёт


In [36]:
%pip install duckdb


Note: you may need to restart the kernel to use updated packages.


In [37]:
DUCKDB_PATH = DATA_PATH.parent / "step_duckdb.py"


#### Навыки и пары: DuckDB


In [38]:
import subprocess
import sys

command = [sys.executable, str(DUCKDB_PATH), MR_INPUT, "1", "2",
           "--min-salaries", str(MIN_SALARIES),
           "--unknown-gross-as-net", str(int(UNKNOWN_GROSS_AS_NET))]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding="utf-8") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("DuckDB завершился с ошибкой — см. вывод выше")



Топ стеков K=1:
('1с программирование', 9412, 5405, 130500.0)
('1с:предприятие 8', 4837, 3025, 130500.0)
('1с:бухгалтерия', 3586, 2142, 130000.0)
('1с:управление торговлей', 2856, 1850, 130500.0)
('обновление конфигурации 1с', 2486, 1501, 130500.0)

Топ стеков K=2:
('1с программирование|1с:предприятие 8', 4084, 2574, 130500.0)
('1с программирование|1с:бухгалтерия', 2884, 1757, 130000.0)
('1с программирование|1с:управление торговлей', 2279, 1537, 130500.0)
('1с программирование|обновление конфигурации 1с', 2209, 1340, 130500.0)
('1с:бухгалтерия|1с:предприятие 8', 1931, 1274, 130500.0)

DuckDB (лимит 2GB): 40.12 с; пик памяти 0.91 ГиБ (private bytes)
K=1: стеков 2,181, вхождений 68,363, зарплат 40,486, сумма медиан 15,742,805.0
K=2: стеков 30,224, вхождений 231,925, зарплат 143,228, сумма медиан 82,114,972.0
Результаты: C:\Users\chuni\PycharmProjects\vacancies_hh\duckdb_work\20261019-135108-1103e8


#### Тройки: DuckDB


In [39]:
import subprocess
import sys

command = [sys.executable, str(DUCKDB_PATH), MR_INPUT, "3",
           "--min-salaries", str(MIN_SALARIES),
           "--unknown-gross-as-net", str(int(UNKNOWN_GROSS_AS_NET))]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding="utf-8") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("DuckDB завершился с ошибкой — см. вывод выше")



Топ стеков K=3:
('1с программирование|1с:бухгалтерия|1с:предприятие 8', 1611, 1070, 130500.0)
('1с программирование|1с:предприятие 8|1с:управление торговлей', 1308, 941, 130500.0)
('1с программирование|1с:предприятие 8|обновление конфигурации 1с', 1254, 836, 130500.0)
('1с программирование|1с:бухгалтерия|1с:управление торговлей', 1163, 812, 130000.0)
('1с программирование|1с:бухгалтерия|1с:зарплата и управление персоналом', 1053, 676, 130500.0)

DuckDB (лимит 2GB): 12.91 с; пик памяти 1.12 ГиБ (private bytes)
K=3: стеков 195,046, вхождений 694,183, зарплат 444,603, сумма медиан 166,839,902.0
Результаты: C:\Users\chuni\PycharmProjects\vacancies_hh\duckdb_work\20261019-135312-d33fe9


#### Верификация, три раунда профилирования и проверка отказа


In [40]:
COMPARE_PATH = DATA_PATH.parent / "compare_methods.py"


In [41]:
import importlib.util
import pandas as pd

spec = importlib.util.spec_from_file_location("compare_methods", COMPARE_PATH)
comparison = importlib.util.module_from_spec(spec)
spec.loader.exec_module(comparison)
checks = pd.DataFrame(comparison.correctness(MR_INPUT))
display(checks)
assert checks["Совпало"].all(), "Есть расхождения — см. таблицу"
print("Все ключи, частоты и медианы MapReduce и DuckDB совпали")
checks.to_csv(DATA_PATH.parent / "comparison_correctness.csv", index=False, encoding="utf-8-sig")


,K,Стеков MapReduce,Стеков DuckDB,Вхождений MapReduce,Вхождений DuckDB,Несовпавших ключей/метрик,Совпало
0,1,2181,2181,68363,68363,0,True
1,2,30224,30224,231925,231925,0,True
2,3,195046,195046,694183,694183,0,True


Все ключи, частоты и медианы MapReduce и DuckDB совпали


#### Три раунда каждого метода


In [42]:
import sys
PROJECT_DIR = DATA_PATH.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
CHECKS_PATH = PROJECT_DIR / "lab_checks.py"


In [44]:
import importlib
import compare_methods
importlib.reload(compare_methods)
import lab_checks
importlib.reload(lab_checks)
assert MR_INPUT == "s3://dataset/it_vacancies_2024.csv"
profile_proof = None
profile_proof = lab_checks.profile_methods(
    MR_INPUT, rounds=3, minimum=MIN_SALARIES, unknown=UNKNOWN_GROSS_AS_NET)



РАУНД 1/3; MapReduce; Навыки + пары
Источник: s3://dataset/it_vacancies_2024.csv; размер 847,619,384 байт
CSV проверен: 229388 записей, переносов внутри записей нет
K=1: map-задач 13, вхождений 68,363, строк shuffle 68,363, shuffle 2.6 МиБ, редьюсеров 8
  1с программирование: вакансий 9412, зарплат 5405, медиана 130500.0
  1с:предприятие 8: вакансий 4837, зарплат 3025, медиана 130500.0
  1с:бухгалтерия: вакансий 3586, зарплат 2142, медиана 130000.0
  1с:управление торговлей: вакансий 2856, зарплат 1850, медиана 130500.0
  обновление конфигурации 1с: вакансий 2486, зарплат 1501, медиана 130500.0
K=2: map-задач 13, вхождений 231,925, строк shuffle 231,925, shuffle 15.9 МиБ, редьюсеров 8
  1с программирование|1с:предприятие 8: вакансий 4084, зарплат 2574, медиана 130500.0
  1с программирование|1с:бухгалтерия: вакансий 2884, зарплат 1757, медиана 130000.0
  1с программирование|1с:управление торговлей: вакансий 2279, зарплат 1537, медиана 130500.0
  1с программирование|обновление конфигура

In [45]:
# Сырые измерения всех трёх раундов, без примеси бонусных и отказных запусков
measurements = pd.read_csv(profile_proof["measurements"])
profile_checks = pd.read_csv(profile_proof["checks"])
assert measurements["Источник"].eq(MR_INPUT).all()
assert measurements["ETag"].nunique() == 1
assert profile_checks["Совпало"].all()
assert measurements.groupby(["Метод", "Расчёт"])["Раунд"].nunique().eq(3).all()
display(measurements[["Раунд", "Метод", "Расчёт", "Время полного запуска, с",
                      "Пик памяти, ГиБ", "Метрика памяти"]].round(3))
display(profile_checks)

performance_summary = measurements.groupby(["Метод", "Расчёт"], sort=False).agg(
    Повторов=("Раунд", "count"),
    Медиана_времени_с=("Время полного запуска, с", "median"),
    Минимум_времени_с=("Время полного запуска, с", "min"),
    Максимум_времени_с=("Время полного запуска, с", "max"),
    Медиана_пика_ГиБ=("Пик памяти, ГиБ", "median"),
    Максимальный_пик_ГиБ=("Пик памяти, ГиБ", "max"),
).round(3)
display(performance_summary)
measurements.to_csv(DATA_PATH.parent / "comparison_performance.csv", index=False, encoding="utf-8-sig")
performance_summary.to_csv(DATA_PATH.parent / "comparison_performance_summary.csv", encoding="utf-8-sig")


,Раунд,Метод,Расчёт,"Время полного запуска, с","Пик памяти, ГиБ",Метрика памяти
0,1,MapReduce,Навыки + пары,30.901,0.904,private bytes
1,1,DuckDB,Навыки + пары,69.150,0.921,private bytes
2,1,MapReduce,Тройки,29.268,0.842,private bytes
3,1,DuckDB,Тройки,35.926,1.115,private bytes
4,2,DuckDB,Навыки + пары,39.818,0.922,private bytes
5,2,MapReduce,Навыки + пары,52.969,0.779,private bytes
6,2,DuckDB,Тройки,15.772,1.122,private bytes
7,2,MapReduce,Тройки,23.521,0.791,private bytes
8,3,MapReduce,Навыки + пары,67.741,0.842,private bytes
9,3,DuckDB,Навыки + пары,42.136,0.906,private bytes


,Раунд,K,Расхождений,Совпало
0,1,1,0,True
1,1,2,0,True
2,1,3,0,True
3,2,1,0,True
4,2,2,0,True
5,2,3,0,True
6,3,1,0,True
7,3,2,0,True
8,3,3,0,True


,,Повторов,Медиана_времени_с,Минимум_времени_с,Максимум_времени_с,Медиана_пика_ГиБ,Максимальный_пик_ГиБ
Метод,Расчёт,,,,,,
MapReduce,Навыки + пары,3,52.969,30.901,67.741,0.842,0.904
DuckDB,Навыки + пары,3,42.136,39.818,69.150,0.921,0.922
MapReduce,Тройки,3,29.268,23.521,42.377,0.791,0.842
DuckDB,Тройки,3,24.015,15.772,35.926,1.115,1.122


#### Отказ mapper и повторная попытка


In [46]:
failure_evidence = pd.DataFrame(lab_checks.demonstrate_failure(
    MR_INPUT, minimum=MIN_SALARIES, unknown=UNKNOWN_GROSS_AS_NET))
display(failure_evidence)
assert failure_evidence["Совпало с запуском без отказа"].all()


Источник: s3://dataset/it_vacancies_2024.csv; размер 847,619,384 байт
CSV проверен: 229388 записей, переносов внутри записей нет
Map 0: попытка 1 не удалась
K=1: map-задач 13, вхождений 68,363, строк shuffle 68,363, shuffle 2.6 МиБ, редьюсеров 8
  1с программирование: вакансий 9412, зарплат 5405, медиана 130500.0
  1с:предприятие 8: вакансий 4837, зарплат 3025, медиана 130500.0
  1с:бухгалтерия: вакансий 3586, зарплат 2142, медиана 130000.0
  1с:управление торговлей: вакансий 2856, зарплат 1850, медиана 130500.0
  обновление конфигурации 1с: вакансий 2486, зарплат 1501, медиана 130500.0
Map 0: попытка 1 не удалась
K=2: map-задач 13, вхождений 231,925, строк shuffle 231,925, shuffle 15.9 МиБ, редьюсеров 8
  1с программирование|1с:предприятие 8: вакансий 4084, зарплат 2574, медиана 130500.0
  1с программирование|1с:бухгалтерия: вакансий 2884, зарплат 1757, медиана 130000.0
  1с программирование|1с:управление торговлей: вакансий 2279, зарплат 1537, медиана 130500.0
  1с программирование|о

,K,Попыток Map 0,Стеков,Совпало с запуском без отказа,Папка
0,1,2,2181,True,C:\Users\chuni\PycharmProjects\vacancies_hh\mr...
1,2,2,30224,True,C:\Users\chuni\PycharmProjects\vacancies_hh\mr...
2,3,2,195046,True,C:\Users\chuni\PycharmProjects\vacancies_hh\mr...


#### Результаты обратно в RustFS: бакет dataset


In [47]:
from urllib.parse import urlsplit
assert urlsplit(MR_INPUT).netloc == "dataset", "В этой работе используем бакет dataset"
published_files = pd.DataFrame(lab_checks.publish_results(
    MR_INPUT, minimum=MIN_SALARIES, unknown=UNKNOWN_GROSS_AS_NET))
display(published_files[["K", "Метод", "Объект", "Байт", "Проверено скачиванием"]])
assert published_files["Проверено скачиванием"].all()
assert published_files.query("Метод == 'MapReduce'").groupby("K").size().eq(8).all()


Проверен s3://dataset/results/20261019-141651-2a74fb/k1/part-00000 (12,194 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k1/part-00001 (11,089 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k1/part-00002 (11,032 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k1/part-00003 (12,446 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k1/part-00004 (11,415 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k1/part-00005 (12,669 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k1/part-00006 (11,597 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k1/part-00007 (10,513 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/duckdb/k1/stacks_1.parquet (48,391 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k2/part-00000 (249,332 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k2/part-00001 (264,916 байт)
Проверен s3://dataset/results/20261019-141651-2a74fb/k2/part-00002 (258,867 байт)
Проверен s3:

,K,Метод,Объект,Байт,Проверено скачиванием
0,1,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k1...,12194,True
1,1,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k1...,11089,True
2,1,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k1...,11032,True
3,1,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k1...,12446,True
4,1,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k1...,11415,True
5,1,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k1...,12669,True
6,1,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k1...,11597,True
7,1,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k1...,10513,True
8,1,DuckDB,s3://dataset/results/20261019-141651-2a74fb/du...,48391,True
9,2,MapReduce,s3://dataset/results/20261019-141651-2a74fb/k2...,249332,True


#### Связь сочетаний навыков с зарплатой


In [48]:
ANALYSIS_PATH = DATA_PATH.parent / "vacancy_analysis.py"


In [49]:
import json
import importlib
from pathlib import Path
import duckdb
import pandas as pd
import compare_methods
import step_duckdb
import lab_checks

importlib.reload(compare_methods)
importlib.reload(step_duckdb)
importlib.reload(lab_checks)
verification = pd.DataFrame(compare_methods.correctness(MR_INPUT))
display(verification)
assert verification["Совпало"].all()
source_now = lab_checks.source_guard(MR_INPUT)
sql_results = {}
for k in (1, 2, 3):
    info = json.loads((DATA_PATH.parent / "duckdb_work" / f"latest-k{k}.json").read_text(encoding="utf-8"))
    report = json.loads(Path(info["summary"]).read_text(encoding="utf-8"))
    lab_checks.check_report(report, source_now, MIN_SALARIES, UNKNOWN_GROSS_AS_NET)
    sql_results[k] = info
print("Проверено: все результаты совпали и относятся к текущему объекту RustFS")


,K,Стеков MapReduce,Стеков DuckDB,Вхождений MapReduce,Вхождений DuckDB,Несовпавших ключей/метрик,Совпало
0,1,2181,2181,68363,68363,0,True
1,2,30224,30224,231925,231925,0,True
2,3,195046,195046,694183,694183,0,True


Проверено: все результаты совпали и относятся к текущему объекту RustFS


In [50]:
# Загружаем полные результаты K=1,2,3: база выбирается до фильтра по зарплатам
import vacancy_analysis
importlib.reload(vacancy_analysis)
with duckdb.connect() as rank_connection:
    result_tables = {
        k: rank_connection.execute(
            "SELECT stack, vacancies, salaries, median_net FROM read_parquet(?)",
            [info["parquet"]]).df()
        for k, info in sql_results.items()
    }
salary_comparisons = {
    k: vacancy_analysis.salary_lifts(result_tables, k, MIN_SALARIES)
    for k in (2, 3)
}

salary_columns = {
    "stack": "Сочетание", "vacancies": "Вакансий", "salaries": "С зарплатой",
    "median_net": "Медиана сочетания, ₽", "base_stack": "База",
    "base_vacancies": "Вакансий в базе", "base_salaries": "Зарплат в базе",
    "base_median": "Медиана базы, ₽", "added": "Добавленный навык",
    "delta": "Разница, ₽", "delta_pct": "Разница, %",
}
def salary_display(table):
    result = table.copy()
    for column in ("stack", "base_stack"):
        result[column] = result[column].map(vacancy_analysis.decode_stack)
    return result.rename(columns=salary_columns).round(1)

for k, table in salary_comparisons.items():
    print(f"K={k}: {len(table):,} сочетаний с достаточной зарплатной выборкой и базой")
    table.to_csv(DATA_PATH.parent / f"salary_comparisons_k{k}.csv", index=False, encoding="utf-8-sig")


K=2: 618 сочетаний с достаточной зарплатной выборкой и базой
K=3: 1,234 сочетаний с достаточной зарплатной выборкой и базой


In [51]:
salary_rankings = {}
for k, kind in ((2, "Пары"), (3, "Тройки")):
    table = salary_comparisons[k]
    high = table.loc[table["delta"].gt(0)].sort_values(
        ["delta", "salaries", "stack"], ascending=[False, False, True]).head(10)
    low = table.loc[table["delta"].lt(0)].sort_values(
        ["delta", "salaries", "stack"], ascending=[True, False, True]).head(10)
    salary_rankings[k] = {"выше": high, "ниже": low}
    print(f"\n{kind}: наиболее высокая медиана относительно своей базы")
    display(salary_display(high))
    print(f"{kind}: наиболее низкая медиана относительно своей базы")
    display(salary_display(low))
    for direction, result in (("higher", high), ("lower", low)):
        salary_display(result).to_csv(DATA_PATH.parent / f"salary_{direction}_k{k}.csv",
                                     index=False, encoding="utf-8-sig")



Пары: наиболее высокая медиана относительно своей базы


,Сочетание,Вакансий,С зарплатой,"Медиана сочетания, ₽",База,Вакансий в базе,Зарплат в базе,"Медиана базы, ₽",Добавленный навык,"Разница, ₽","Разница, %"
544,1с программирование + java,46,32,270000.0,1с программирование,9412,5405,130500.0,java,139500.0,106.9
168,java + sql,44,34,270000.0,sql,910,536,150000.0,java,120000.0,80.0
362,базы данных + скд,62,52,200000.0,базы данных,810,528,130500.0,скд,69500.0,53.3
208,1с:предприятие 8 + 1с:предприятие:бухгалтерски...,77,46,200000.0,1с:предприятие 8,4837,3025,130500.0,1с:предприятие:бухгалтерский учет,69500.0,53.3
262,erp + обновление конфигурации 1с,62,38,200000.0,обновление конфигурации 1с,2486,1501,130500.0,erp,69500.0,53.3
325,1с программирование + json,76,31,200000.0,1с программирование,9412,5405,130500.0,json,69500.0,53.3
48,1с программирование + wms,93,61,191400.0,1с программирование,9412,5405,130500.0,wms,60900.0,46.7
125,1с:предприятие + 1с:предприятие:бухгалтерский ...,50,30,174000.0,1с:предприятие,1627,1013,120000.0,1с:предприятие:бухгалтерский учет,54000.0,45.0
90,скд + создание конфигурации 1с,105,84,190700.0,создание конфигурации 1с,1576,1028,140000.0,скд,50700.0,36.2
327,erp-системы на базе 1с + sql,144,68,200000.0,erp-системы на базе 1с,2302,1127,150000.0,sql,50000.0,33.3


Пары: наиболее низкая медиана относительно своей базы


,Сочетание,Вакансий,С зарплатой,"Медиана сочетания, ₽",База,Вакансий в базе,Зарплат в базе,"Медиана базы, ₽",Добавленный навык,"Разница, ₽","Разница, %"
80,1с + создание конфигурации 1с,40,30,60900.0,создание конфигурации 1с,1576,1028,140000.0,1с,-79100.0,-56.5
538,базы данных + работа с большим объемом информации,236,179,52200.0,базы данных,810,528,130500.0,работа с большим объемом информации,-78300.0,-60.0
473,базы данных + работа с базами данных,242,159,52200.0,базы данных,810,528,130500.0,работа с базами данных,-78300.0,-60.0
177,1с программирование + пользователь пк,116,89,52200.0,1с программирование,9412,5405,130500.0,пользователь пк,-78300.0,-60.0
607,1с:управление производственным предприятием + ...,56,30,69600.0,1с:управление производственным предприятием,1370,766,140000.0,работа с большим объемом информации,-70400.0,-50.3
565,erp-системы на базе 1с + обучение и развитие,62,39,80000.0,erp-системы на базе 1с,2302,1127,150000.0,обучение и развитие,-70000.0,-46.7
1,1с + 1с:комплексная автоматизация,47,36,60900.0,1с:комплексная автоматизация,1098,684,130000.0,1с,-69100.0,-53.2
51,1с:crm проф + 1с:комплексная автоматизация,35,33,60900.0,1с:комплексная автоматизация,1098,684,130000.0,1с:crm проф,-69100.0,-53.2
231,1с:зарплата и управление персоналом + автомати...,80,51,87000.0,1с:зарплата и управление персоналом,2292,1270,150000.0,автоматизация процессов,-63000.0,-42.0
564,1с:бухгалтерия + администрирование,49,31,69600.0,1с:бухгалтерия,3586,2142,130000.0,администрирование,-60400.0,-46.5



Тройки: наиболее высокая медиана относительно своей базы


,Сочетание,Вакансий,С зарплатой,"Медиана сочетания, ₽",База,Вакансий в базе,Зарплат в базе,"Медиана базы, ₽",Добавленный навык,"Разница, ₽","Разница, %"
968,1с программирование + java + sql,38,31,270000.0,1с программирование + sql,658,375,150000.0,java,120000.0,80.0
208,1с:предприятие + базы данных + работа с больши...,55,36,150000.0,базы данных + работа с большим объемом информации,236,179,52200.0,1с:предприятие,97800.0,187.4
811,1с:комплексная автоматизация + 1с:предприятие ...,63,47,217500.0,1с:комплексная автоматизация + 1с:предприятие 8,577,381,120000.0,1с:управление предприятием,97500.0,81.2
826,1с:комплексная автоматизация + 1с:управление п...,40,32,217500.0,1с:комплексная автоматизация + создание конфиг...,172,134,120000.0,1с:управление предприятием,97500.0,81.2
464,1с:управление торговлей + работа с базами данн...,43,32,130500.0,работа с базами данных + работа с большим объе...,234,158,52200.0,1с:управление торговлей,78300.0,150.0
885,1с:комплексная автоматизация + 1с:управление п...,101,55,200000.0,1с:комплексная автоматизация + 1с:управление т...,574,382,130000.0,1с:управление предприятием,70000.0,53.8
621,1с:бухгалтерия + 1с:зарплата и кадры + 1с:пред...,56,30,210000.0,1с:бухгалтерия + 1с:зарплата и кадры,737,497,140000.0,1с:предприятие:бухгалтерский учет,70000.0,50.0
181,1с:предприятие + базы данных + работа с базами...,55,37,121800.0,базы данных + работа с базами данных,242,159,52200.0,1с:предприятие,69600.0,133.3
759,1с программирование + базы данных + скд,61,51,200000.0,1с программирование + базы данных,734,481,130500.0,скд,69500.0,53.3
869,1с программирование + 1с:предприятие 8 + 1с:пр...,75,44,200000.0,1с программирование + 1с:предприятие 8,4084,2574,130500.0,1с:предприятие:бухгалтерский учет,69500.0,53.3


Тройки: наиболее низкая медиана относительно своей базы


,Сочетание,Вакансий,С зарплатой,"Медиана сочетания, ₽",База,Вакансий в базе,Зарплат в базе,"Медиана базы, ₽",Добавленный навык,"Разница, ₽","Разница, %"
661,1с:зарплата и управление персоналом + автомати...,52,34,26100.0,1с:зарплата и управление персоналом + обновлен...,520,342,150000.0,автоматизация процессов,-123900.0,-82.6
938,1с:предприятие + 1с:розница + обновление конфи...,39,34,28050.0,1с:предприятие + обновление конфигурации 1с,479,334,130500.0,1с:розница,-102450.0,-78.5
658,1с:розница + erp-системы на базе 1с + обновлен...,50,40,52200.0,erp-системы на базе 1с + обновление конфигурац...,654,374,148950.0,1с:розница,-96750.0,-65.0
1131,erp-системы на базе 1с + автоматизация процесс...,66,40,52200.0,erp-системы на базе 1с + обновление конфигурац...,654,374,148950.0,автоматизация процессов,-96750.0,-65.0
1161,1с:бухгалтерия + erp-системы на базе 1с + авто...,48,33,52200.0,1с:бухгалтерия + erp-системы на базе 1с,740,407,147900.0,автоматизация процессов,-95700.0,-64.7
505,1с:предприятие + erp-системы на базе 1с + авто...,48,32,34800.0,1с:предприятие + erp-системы на базе 1с,420,265,130500.0,автоматизация процессов,-95700.0,-73.3
652,1с:бухгалтерия + базы данных + работа с базами...,105,63,52200.0,1с:бухгалтерия + базы данных,289,205,134850.0,работа с базами данных,-82650.0,-61.3
923,1с:предприятие + 1с:управление торговлей + авт...,54,34,50000.0,1с:предприятие + 1с:управление торговлей,547,344,130000.0,автоматизация процессов,-80000.0,-61.5
878,1с + 1с программирование + создание конфигурац...,40,30,60900.0,1с программирование + создание конфигурации 1с,1447,944,140000.0,1с,-79100.0,-56.5
1133,1с программирование + базы данных + работа с б...,230,173,52200.0,1с программирование + базы данных,734,481,130500.0,работа с большим объемом информации,-78300.0,-60.0


#### Наблюдения и проверка объяснений снижения



In [55]:
from IPython.display import Markdown, display

if "analysis_con" in globals():
    analysis_con.close()

analysis_con, analysis_source = vacancy_analysis.open_analysis(
    MR_INPUT, UNKNOWN_GROSS_AS_NET
)

assert all(
    analysis_source.get(field) == source_now.get(field)
    for field in lab_checks.IDENTITY_FIELDS
)

for k in (2, 3):
    for direction, ranking in salary_rankings[k].items():
        if ranking.empty:
            continue

        best = ranking.iloc[0]

        if direction == "ниже":
            for dimension, label in (("experience", "Опыт"), ("area", "Территория")):
                composition = vacancy_analysis.context_table(
                    analysis_con,
                    best["stack"],
                    best["base_stack"],
                    dimension,
                    MIN_SALARIES,
                )

                # --- вывод таблицы ---
                print(f"K={k}. Проверка состава зарплатной выборки: {label}")
                display(
                    composition.rename(
                        columns={
                            "group_name": "Группа",
                            "base_paid": "Зарплат в базе",
                            "combo_paid": "Зарплат в сочетании",
                            "base_median": "Медиана базы, ₽",
                            "combo_median": "Медиана сочетания, ₽",
                            "base_share_pct": "Доля в базе, %",
                            "combo_share_pct": "Доля в сочетании, %",
                            "share_delta_pp": "Разница долей, п. п.",
                            "delta": "Разница медиан, ₽",
                        }
                    ).round(1)
                )

                composition.to_csv(
                    DATA_PATH.parent / f"salary_context_k{k}_{dimension}.csv",
                    index=False,
                    encoding="utf-8-sig",
                )

K=2. Проверка состава зарплатной выборки: Опыт


,Группа,Зарплат в базе,Зарплат в сочетании,"Медиана базы, ₽","Медиана сочетания, ₽","Доля в базе, %","Доля в сочетании, %","Разница долей, п. п.","Разница медиан, ₽"
0,Нет опыта,92,21,60900.0,NaN,8.9,70.0,61.1,NaN
1,От 1 года до 3 лет,527,6,120000.0,NaN,51.3,20.0,-31.3,NaN
2,Более 6 лет,40,2,180000.0,NaN,3.9,6.7,2.8,NaN
3,От 3 до 6 лет,369,1,165000.0,NaN,35.9,3.3,-32.6,NaN


K=2. Проверка состава зарплатной выборки: Территория


,Группа,Зарплат в базе,Зарплат в сочетании,"Медиана базы, ₽","Медиана сочетания, ₽","Доля в базе, %","Доля в сочетании, %","Разница долей, п. п.","Разница медиан, ₽"
0,Москва,232,8,180000.0,NaN,22.6,26.7,4.1,NaN
1,Санкт-Петербург,99,4,150000.0,NaN,9.6,13.3,3.7,NaN
2,Екатеринбург,67,3,150000.0,NaN,6.5,10.0,3.5,NaN
3,Самара,19,2,NaN,NaN,1.8,6.7,4.8,NaN
4,Волгоград,11,1,NaN,NaN,1.1,3.3,2.3,NaN
...,...,...,...,...,...,...,...,...,...
115,Челябинск,17,0,NaN,NaN,1.7,0.0,-1.7,NaN
116,Череповец,1,0,NaN,NaN,0.1,0.0,-0.1,NaN
117,Чита,1,0,NaN,NaN,0.1,0.0,-0.1,NaN
118,Южно-Сахалинск,3,0,NaN,NaN,0.3,0.0,-0.3,NaN


K=3. Проверка состава зарплатной выборки: Опыт


,Группа,Зарплат в базе,Зарплат в сочетании,"Медиана базы, ₽","Медиана сочетания, ₽","Доля в базе, %","Доля в сочетании, %","Разница долей, п. п.","Разница медиан, ₽"
0,Нет опыта,28,16,NaN,NaN,8.2,47.1,38.9,NaN
1,От 1 года до 3 лет,183,9,130500.0,NaN,53.5,26.5,-27.0,NaN
2,От 3 до 6 лет,122,8,174000.0,NaN,35.7,23.5,-12.1,NaN
3,Более 6 лет,9,1,NaN,NaN,2.6,2.9,0.3,NaN


K=3. Проверка состава зарплатной выборки: Территория


,Группа,Зарплат в базе,Зарплат в сочетании,"Медиана базы, ₽","Медиана сочетания, ₽","Доля в базе, %","Доля в сочетании, %","Разница долей, п. п.","Разница медиан, ₽"
0,Москва,84,4,200000.0,NaN,24.6,11.8,-12.8,NaN
1,Новосибирск,13,3,NaN,NaN,3.8,8.8,5.0,NaN
2,Владимир,3,2,NaN,NaN,0.9,5.9,5.0,NaN
3,Екатеринбург,8,2,NaN,NaN,2.3,5.9,3.5,NaN
4,Симферополь,8,2,NaN,NaN,2.3,5.9,3.5,NaN
...,...,...,...,...,...,...,...,...,...
85,Элиста,1,0,NaN,NaN,0.3,0.0,-0.3,NaN
86,Южно-Сахалинск,1,0,NaN,NaN,0.3,0.0,-0.3,NaN
87,Якутск,1,0,NaN,NaN,0.3,0.0,-0.3,NaN
88,Янино-1,1,0,NaN,NaN,0.3,0.0,-0.3,NaN


#### Cравнение трёх территорий

In [56]:
# Используем подготовленные из RustFS вакансии этапа 5
if "analysis_con" not in globals():
    analysis_con, analysis_source = vacancy_analysis.open_analysis(MR_INPUT, UNKNOWN_GROSS_AS_NET)
assert step_duckdb.object_identity(MR_INPUT) == analysis_source, "Объект изменился — повторите этап 5"
regional_summary_raw, regional_results = vacancy_analysis.regional_tables(analysis_con, MIN_SALARIES)
assert step_duckdb.object_identity(MR_INPUT) == analysis_source

regional_summary = regional_summary_raw.rename(columns={
    "area": "Территория", "vacancies": "Вакансий", "salaries": "С зарплатой",
    "with_skills": "С навыками", "median_net": "Медиана net, ₽",
    "paid_pct": "С зарплатой, %", "skills_pct": "С навыками, %"})
display(regional_summary.round(1))
regional_summary.to_csv(DATA_PATH.parent / "regional_summary.csv", index=False, encoding="utf-8-sig")
for k, table in regional_results.items():
    table.to_csv(DATA_PATH.parent / f"regional_all_k{k}.csv", index=False, encoding="utf-8-sig")


,Территория,Вакансий,С зарплатой,С навыками,"Медиана net, ₽","С зарплатой, %","С навыками, %"
0,Москва,5248,2284,3456,180000.0,43.5,65.9
1,Санкт-Петербург,1703,896,1159,150000.0,52.6,68.1
2,Екатеринбург,884,546,595,130500.0,61.8,67.3


In [57]:
regional_top = {}
regional_premium = {}
regional_columns = {
    "Сочетание": "Сочетание", "vacancies": "Вакансий с сочетанием",
    "share_pct": "Доля вакансий, %", "salaries": "С зарплатой",
    "median_net": "Медиана сочетания, ₽", "region_median": "Медиана территории, ₽",
    "delta_region": "Разница с территорией, ₽",
}
for k, table in regional_results.items():
    regional_top[k] = (table.sort_values(["area", "vacancies", "stack"],
        ascending=[True, False, True]).groupby("area", sort=False).head(5))
    premium = table.loc[table["share_pct"].ge(5) & table["salaries"].ge(MIN_SALARIES)
                        & table["delta_region"].gt(0)]
    regional_premium[k] = (premium.sort_values(["area", "median_net", "vacancies", "stack"],
        ascending=[True, False, False, True]).groupby("area", sort=False).head(3))
    regional_top[k].to_csv(DATA_PATH.parent / f"regional_top_frequency_k{k}.csv", index=False, encoding="utf-8-sig")
    regional_premium[k].to_csv(DATA_PATH.parent / f"regional_top_salary_k{k}.csv", index=False, encoding="utf-8-sig")

for territory in regional_summary_raw["area"]:
    for k, title in ((1, "Навыки"), (2, "Пары"), (3, "Тройки")):
        print(f"\n{territory}. {title}: топ-5 по частоте")
        frequent = regional_top[k].loc[regional_top[k]["area"].eq(territory)]
        display(frequent[list(regional_columns)].rename(columns=regional_columns).round(1))
        print(f"{title}: топ-3 по медиане при доле ≥5%, зарплат ≥{MIN_SALARIES} и медиане выше местной")
        higher_paid = regional_premium[k].loc[regional_premium[k]["area"].eq(territory)]
        if higher_paid.empty:
            print("При заданных порогах подходящих сочетаний нет")
        else:
            display(higher_paid[list(regional_columns)].rename(columns=regional_columns).round(1))



Москва. Навыки: топ-5 по частоте


,Сочетание,Вакансий с сочетанием,"Доля вакансий, %",С зарплатой,"Медиана сочетания, ₽","Медиана территории, ₽","Разница с территорией, ₽"
1573,1с программирование,2506,47.8,1180,180000.0,180000.0,0.0
542,1с:предприятие 8,1260,24.0,657,191400.0,180000.0,11400.0
1128,1с:бухгалтерия,974,18.6,500,191400.0,180000.0,11400.0
537,1с:управление торговлей,784,14.9,437,180000.0,180000.0,0.0
1578,erp-системы на базе 1с,746,14.2,333,200000.0,180000.0,20000.0


Навыки: топ-3 по медиане при доле ≥5%, зарплат ≥30 и медиане выше местной


,Сочетание,Вакансий с сочетанием,"Доля вакансий, %",С зарплатой,"Медиана сочетания, ₽","Медиана территории, ₽","Разница с территорией, ₽"
1578,erp-системы на базе 1с,746,14.2,333,200000.0,180000.0,20000.0
1580,1с:зарплата и управление персоналом,679,12.9,305,200000.0,180000.0,20000.0
1579,обновление конфигурации 1с,601,11.5,310,200000.0,180000.0,20000.0



Москва. Пары: топ-5 по частоте


...

,Сочетание,Вакансий с сочетанием,"Доля вакансий, %",С зарплатой,"Медиана сочетания, ₽","Медиана территории, ₽","Разница с территорией, ₽"
4390,1с программирование + обновление конфигурации 1с,116,13.1,82,150000.0,130500.0,19500.0
287,1с программирование + 1с:управление торговлей,104,11.8,64,150000.0,130500.0,19500.0
4269,1с программирование + erp-системы на базе 1с,92,10.4,53,150000.0,130500.0,19500.0



Екатеринбург. Тройки: топ-5 по частоте


,Сочетание,Вакансий с сочетанием,"Доля вакансий, %",С зарплатой,"Медиана сочетания, ₽","Медиана территории, ₽","Разница с территорией, ₽"
112598,1с программирование + 1с:бухгалтерия + 1с:пред...,80,9.0,48,138950.0,130500.0,8450.0
75805,1с программирование + 1с:предприятие 8 + обнов...,69,7.8,52,150000.0,130500.0,19500.0
62571,1с программирование + 1с:предприятие 8 + 1с:уп...,63,7.1,41,150000.0,130500.0,19500.0
27900,1с программирование + 1с:бухгалтерия + 1с:упра...,62,7.0,33,150000.0,130500.0,19500.0
104349,1с программирование + 1с:документооборот + 1с:...,50,5.7,35,150000.0,130500.0,19500.0


Тройки: топ-3 по медиане при доле ≥5%, зарплат ≥30 и медиане выше местной


,Сочетание,Вакансий с сочетанием,"Доля вакансий, %",С зарплатой,"Медиана сочетания, ₽","Медиана территории, ₽","Разница с территорией, ₽"
75805,1с программирование + 1с:предприятие 8 + обнов...,69,7.8,52,150000.0,130500.0,19500.0
62571,1с программирование + 1с:предприятие 8 + 1с:уп...,63,7.1,41,150000.0,130500.0,19500.0
27900,1с программирование + 1с:бухгалтерия + 1с:упра...,62,7.0,33,150000.0,130500.0,19500.0


#### Оптимизации MapReduce при чтении из RustFS


In [58]:
BENCHMARK_PATH = DATA_PATH.parent / "benchmark_rustfs.py"


In [59]:
import os
import subprocess
import sys
from pathlib import Path

assert MR_INPUT.startswith("s3://"), "Для итогового сравнения нужен объект RustFS"
assert os.environ.get("AWS_ACCESS_KEY_ID") and os.environ.get("AWS_SECRET_ACCESS_KEY"), (
    "Сначала выполните ячейку подключения к RustFS")

benchmark_script = DATA_PATH.parent / "benchmark_rustfs.py"
command = [sys.executable, str(benchmark_script), MR_INPUT,
           "--workers", "8", "--rounds", "3"]
with subprocess.Popen(command, cwd=DATA_PATH.parent, env=os.environ.copy(),
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding="utf-8", errors="replace") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Сравнение из RustFS завершилось с ошибкой; см. вывод выше")


Объект: s3://dataset/it_vacancies_2024.csv; 847619384 байт; ETag "be9c1b4b3f57ff0e9771004c2a5fe78b-51"
Раунд 1, Базовый: 38.05 с; пик 1.43 ГиБ; shuffle 86.6 МиБ
Раунд 1, Агрегация: 36.24 с; пик 1.58 ГиБ; shuffle 58.1 МиБ
Раунд 1, ID навыков: 34.81 с; пик 1.41 ГиБ; shuffle 14.9 МиБ
Раунд 1, Агрегация + ID: 35.73 с; пик 1.48 ГиБ; shuffle 12.7 МиБ
Раунд 1, Один проход: 23.93 с; пик 1.28 ГиБ; shuffle 86.6 МиБ
Раунд 1: каждый ключ, частота и медиана K=1,2,3 совпали во всех пяти режимах
Раунд 2, Агрегация: 92.84 с; пик 1.53 ГиБ; shuffle 58.1 МиБ
Раунд 2, ID навыков: 44.34 с; пик 1.53 ГиБ; shuffle 14.9 МиБ
Раунд 2, Агрегация + ID: 40.38 с; пик 1.60 ГиБ; shuffle 12.7 МиБ
Раунд 2, Один проход: 23.45 с; пик 1.40 ГиБ; shuffle 86.6 МиБ
Раунд 2, Базовый: 35.53 с; пик 1.53 ГиБ; shuffle 86.6 МиБ
Раунд 2: каждый ключ, частота и медиана K=1,2,3 совпали во всех пяти режимах
Раунд 3, ID навыков: 37.13 с; пик 1.47 ГиБ; shuffle 14.9 МиБ
Раунд 3, Агрегация + ID: 34.98 с; пик 1.39 ГиБ; shuffle 12.7 МиБ
Раунд

In [60]:
import pandas as pd

bonus_path = DATA_PATH.parent / "rustfs_bonus_benchmark.csv"
bonus = pd.read_csv(bonus_path)
assert set(bonus["Источник"]) == {MR_INPUT}
assert bonus["ETag"].nunique() == 1 and bonus["Размер, байт"].nunique() == 1
assert bonus["Все результаты совпали"].all()
assert bonus["Раунд"].nunique() == 3
assert (bonus.groupby("Раунд")["Вариант"].nunique() == 5).all()

summary_bonus = (bonus.groupby("Вариант", sort=False)
                 .agg({"Время, с": "median", "Пик памяти, ГиБ": "median",
                       "Shuffle, МиБ": "median", "Запусков Map": "median"})
                 .round(2))
baseline_time = summary_bonus.loc["Базовый", "Время, с"]
summary_bonus["Ускорение к базовому, %"] = (
    (1 - summary_bonus["Время, с"] / baseline_time) * 100).round(1)
print("Источник:", bonus["Источник"].iloc[0],
      "| байт:", bonus["Размер, байт"].iloc[0],
      "| ETag:", bonus["ETag"].iloc[0])
display(summary_bonus)


Источник: s3://dataset/it_vacancies_2024.csv | байт: 847619384 | ETag: "be9c1b4b3f57ff0e9771004c2a5fe78b-51"


,"Время, с","Пик памяти, ГиБ","Shuffle, МиБ",Запусков Map,"Ускорение к базовому, %"
Вариант,,,,,
Базовый,35.53,1.48,86.60,39.0,0.0
Агрегация,36.24,1.53,58.07,39.0,-2.0
ID навыков,37.12,1.47,14.86,39.0,-4.5
Агрегация + ID,35.72,1.48,12.72,39.0,-0.5
Один проход,23.45,1.40,86.60,13.0,34.0


#### Эффект одного прохода


In [61]:
paired = bonus[bonus["Вариант"].isin(["Базовый", "Один проход"])].pivot(
    index="Раунд", columns="Вариант", values="Время, с")
paired["Ускорение, %"] = (
    (1 - paired["Один проход"] / paired["Базовый"]) * 100).round(1)
display(paired)
print("Медиана парного ускорения:",
      f"{paired['Ускорение, %'].median():.1f}%")


Вариант,Базовый,Один проход,"Ускорение, %"
Раунд,,,
1,38.052,23.935,37.1
2,35.533,23.448,34.0
3,35.397,21.195,40.1


Медиана парного ускорения: 37.1%
